# Experiment 11b — LSTM Encoder-Decoder with Cross-Attention

This notebook is the LSTM twin of `kp-experiment-11a-gru.ipynb`. **The only architectural
difference is the recurrent cell type**: Bi-GRU → Bi-LSTM in the encoder,
Uni-GRU → Uni-LSTM in the decoder. All other components are identical:

- Same Bahdanau cross-attention (via `nn.MultiheadAttention`)
- Same 168h base lookback with subsequence-crop augmentation
- Same block-C velocity features (`queue_pressure`, `network_pressure`, `shock_risk`)
- Same training recipe (AdamW + CosineAnnealing + MAE loss + gradient clipping)
- Same augmentations (feature dropout, subsequence crop, hard-series oversample)
- **Best config carried from 11a ablation** — no re-ablation needed; LSTM vs GRU is the
  sole comparison variable

The LSTM cell adds a cell state (`c`) alongside the hidden state (`h`), which may help
retain long-range seasonal information over the 168h lookback window. Whether this
translates to a WAPE gain is an empirical question answered by comparing §7 here with
§7 of 11a.

Read `README.md` for joint interpretation with 11a (GRU) and 11c (blend).

---
## §0. Configuration — identical to 11a; best aug config imported

In [2]:
from pathlib import Path

DATA_DIR   = Path("/kaggle/input/datasets/xitij27/dlam-bonus-project-dataset/data")
OUTPUT_DIR = Path("/kaggle/working")
# Local override:
# DATA_DIR   = Path("../data")
# OUTPUT_DIR = Path(".")

TRAIN_PATH     = DATA_DIR / "train.csv"
VAL_INPUT_PATH = DATA_DIR / "validation_input.csv"
VAL_INDEX_PATH = DATA_DIR / "forecast_index_validation.csv"

EXP_ID  = "11b_lstm"
OOF_OUT = OUTPUT_DIR / f"oof_predictions_{EXP_ID}.csv"
VAL_OUT = OUTPUT_DIR / f"validation_predictions_{EXP_ID}.csv"

# ── Reproducibility & CV ──────────────────────────────────────────────────────
SEED         = 42
N_FOLDS      = 3
EVAL_HORIZON = 336

# ── Feature columns ───────────────────────────────────────────────────────────
DOMAIN_FEAT_COLS = [
    "workload_intensity", "demand_forecast", "upstream_quality_forecast",
    "promotion_intensity", "shock_risk", "maintenance_known",
    "unit_reliability_forecast", "queue_pressure_forecast",
    "network_pressure_forecast", "event_load_forecast",
    "service_irregularity_risk_forecast",
]
VELOCITY_COLS = ["queue_pressure_forecast", "network_pressure_forecast", "shock_risk"]
TIME_COLS     = ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "doy_sin", "doy_cos", "is_weekend"]

N_DOMAIN = len(DOMAIN_FEAT_COLS)
N_VELS   = len(VELOCITY_COLS)
N_TIME   = len(TIME_COLS)

# ── Architecture ──────────────────────────────────────────────────────────────
ENCODER_LOOKBACK = 168
CROP_LOOKBACK_LO = 168
CROP_LOOKBACK_HI = 336

EMBED_DIM    = 16
HIDDEN_DIM   = 128
N_ENC_LAYERS = 2
N_DEC_LAYERS = 1
ATTN_HEADS   = 4
DROPOUT      = 0.2

BATCH_SIZE   = 512   # T4 has plenty of VRAM; larger batches keep GPU busy
N_EPOCHS     = 50
PATIENCE     = 7     # saves ~3 wasted epochs on average
LR           = 1e-3
LR_MIN       = 1e-5
WEIGHT_DECAY = 1e-4
GRAD_CLIP    = 1.0
STRIDE       = 24    # one window per day (was 6); 4× smaller dataset → 4× faster epochs
N_WORKERS    = 4     # parallel data loading; stops GPU idling between batches

N_SEEDS   = 5
SEQ_SEEDS = [42, 7, 13, 99, 1234]

# ── Aug config — carried from 11a ablation ────────────────────────────────────
# 11a ablation showed '+aug' was best (aug=True, velocity=False)
USE_AUG      = True
USE_VELOCITY = True
FEAT_DROP_P  = 0.1
HARD_SERIES_OVS = 3

HARD_SERIES_IDS = [
    "unit_073", "unit_058", "unit_066", "unit_056", "unit_039",
    "unit_071", "unit_003", "unit_079", "unit_031", "unit_061",
    "unit_012", "unit_027", "unit_034", "unit_048", "unit_051",
    "unit_063", "unit_069", "unit_077", "unit_082", "unit_090",
]

---
## §1. Setup & Data Loading

In [3]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import RobustScaler

np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

def wape(y_true, y_pred, eps=1e-8):
    return np.sum(np.abs(np.asarray(y_true) - np.asarray(y_pred))) / \
           (np.sum(np.abs(y_true)) + eps) * 100

train_df  = pd.read_csv(TRAIN_PATH,     parse_dates=["timestamp"])
val_input = pd.read_csv(VAL_INPUT_PATH, parse_dates=["timestamp"])
val_index = pd.read_csv(VAL_INDEX_PATH, parse_dates=["timestamp"])

sid_to_code = {sid: i for i, sid in enumerate(sorted(train_df["series_id"].unique()))}
code_to_sid = {v: k for k, v in sid_to_code.items()}
N_SERIES = len(sid_to_code)
print(f"train_df : {train_df.shape}, # series : {N_SERIES}")

PyTorch device: cuda
  GPU: Tesla T4
train_df : (414720, 25), # series : 96


---
## §2. Preprocessing & Sequence Arrays (same as 11a)

In [4]:
def impute_domain_features(df, cols):
    df = df.sort_values(["series_id", "timestamp"]).copy()
    df[cols] = df.groupby("series_id", group_keys=False)[cols].transform(
        lambda x: x.ffill().bfill())
    return df

def add_time_features(df):
    df = df.copy()
    t = pd.to_datetime(df["timestamp"])
    df["hour_sin"]   = np.sin(2 * np.pi * t.dt.hour / 24)
    df["hour_cos"]   = np.cos(2 * np.pi * t.dt.hour / 24)
    df["dow_sin"]    = np.sin(2 * np.pi * t.dt.dayofweek / 7)
    df["dow_cos"]    = np.cos(2 * np.pi * t.dt.dayofweek / 7)
    df["doy_sin"]    = np.sin(2 * np.pi * t.dt.dayofyear / 365)
    df["doy_cos"]    = np.cos(2 * np.pi * t.dt.dayofyear / 365)
    df["is_weekend"] = (t.dt.dayofweek >= 5).astype(float)
    return df

train_df  = impute_domain_features(train_df,  DOMAIN_FEAT_COLS)
train_df  = add_time_features(train_df)
train_df  = train_df.sort_values(["series_id", "timestamp"]).reset_index(drop=True)
val_input = impute_domain_features(val_input, DOMAIN_FEAT_COLS)
val_input = add_time_features(val_input)
val_input = val_input.sort_values(["series_id", "timestamp"]).reset_index(drop=True)

T_PER_SERIES = len(train_df) // N_SERIES
VEL_IDX = [DOMAIN_FEAT_COLS.index(c) for c in VELOCITY_COLS]

SERIES_ARR = {}
for s in range(N_SERIES):
    sid = code_to_sid[s]
    g   = train_df[train_df["series_id"] == sid].sort_values("timestamp")
    cov = g[DOMAIN_FEAT_COLS].to_numpy(dtype=np.float32)
    vel = np.zeros_like(cov[:, VEL_IDX])
    vel[1:] = np.diff(cov[:, VEL_IDX], axis=0)
    SERIES_ARR[s] = dict(
        cov  = cov,
        vel  = vel,
        time = g[TIME_COLS].to_numpy(dtype=np.float32),
        tgt  = g["target"].to_numpy(dtype=np.float32),
        ts   = g["timestamp"].to_numpy(),
    )

VAL_ARR = {}
for s in range(N_SERIES):
    sid = code_to_sid[s]
    g   = val_input[val_input["series_id"] == sid].sort_values("timestamp")
    last_cov = SERIES_ARR[s]["cov"][-1:, VEL_IDX]
    fut_cov_vel = g[VELOCITY_COLS].to_numpy(dtype=np.float32)
    fut_stitch  = np.concatenate([last_cov, fut_cov_vel], axis=0)
    fut_vel     = np.diff(fut_stitch, axis=0)
    VAL_ARR[s]  = dict(
        cov  = g[DOMAIN_FEAT_COLS].to_numpy(dtype=np.float32),
        vel  = fut_vel.astype(np.float32),
        time = g[TIME_COLS].to_numpy(dtype=np.float32),
        ts   = g["timestamp"].to_numpy(),
    )

print(f"T_PER_SERIES={T_PER_SERIES}, series built: {len(SERIES_ARR)}")

T_PER_SERIES=4320, series built: 96


---
## §3. CV Framework & Dataset (same as 11a)

In [5]:
def fold_train_end_idx(k):
    return T_PER_SERIES - (N_FOLDS - k) * EVAL_HORIZON

def fold_test_slice(k):
    start = fold_train_end_idx(k)
    return start, start + EVAL_HORIZON

HARD_CODES = set(sid_to_code.get(s, -1) for s in HARD_SERIES_IDS if s in sid_to_code)

def fit_fold_normalizers(k):
    end_idx = fold_train_end_idx(k)
    all_cov = np.concatenate([SERIES_ARR[s]["cov"][:end_idx] for s in range(N_SERIES)])
    all_vel = np.concatenate([SERIES_ARR[s]["vel"][:end_idx] for s in range(N_SERIES)])
    feat_scaler = RobustScaler().fit(all_cov)
    vel_scaler  = RobustScaler().fit(all_vel)
    mu = np.array([SERIES_ARR[s]["tgt"][:end_idx].mean() for s in range(N_SERIES)], dtype=np.float32)
    sd = np.array([max(float(SERIES_ARR[s]["tgt"][:end_idx].std()), 1e-3) for s in range(N_SERIES)], dtype=np.float32)
    return feat_scaler, vel_scaler, mu, sd


class Seq2SeqDataset(Dataset):
    """
    Velocity for encoder AND decoder is served from a precomputed normalised array
    (no per-sample sklearn calls in __getitem__).
    """
    def __init__(self, fold_k, feat_scaler, vel_scaler, mu, sd,
                 use_aug=False, feat_drop_p=0.0, oversample_factor=1,
                 end_idx_override=None):
        self.feat_scaler  = feat_scaler
        self.vel_scaler   = vel_scaler
        self.mu, self.sd  = mu, sd
        self.use_aug      = use_aug
        self.feat_drop_p  = feat_drop_p
        self.crop_lo      = CROP_LOOKBACK_LO
        self.crop_hi      = CROP_LOOKBACK_HI

        end_idx = end_idx_override if end_idx_override is not None else fold_train_end_idx(fold_k)
        self.samples = []
        for s in range(N_SERIES):
            weight    = oversample_factor if (use_aug and s in HARD_CODES) else 1
            max_start = max(1, end_idx - self.crop_hi - EVAL_HORIZON)
            for start in range(0, max_start, STRIDE):
                for _ in range(weight):
                    self.samples.append((s, start))

        # Precompute ALL normalised arrays once — __getitem__ only slices
        self.normalized = {}
        for s in range(N_SERIES):
            cov_n = np.nan_to_num(feat_scaler.transform(SERIES_ARR[s]["cov"]), nan=0.0).astype(np.float32)
            vel_n = np.nan_to_num(vel_scaler.transform(SERIES_ARR[s]["vel"]), nan=0.0).astype(np.float32)
            tgt_n = ((SERIES_ARR[s]["tgt"] - mu[s]) / sd[s]).astype(np.float32)
            self.normalized[s] = dict(cov=cov_n, vel=vel_n, tgt=tgt_n, time=SERIES_ARR[s]["time"])

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        s, start = self.samples[idx]
        lookback = np.random.randint(self.crop_lo, self.crop_hi + 1) if self.use_aug else self.crop_lo
        arr = self.normalized[s]

        enc_end  = start + lookback
        dec_end  = enc_end + EVAL_HORIZON

        past_cov  = arr["cov"][start:enc_end].copy()   # copy before possible dropout
        past_vel  = arr["vel"][start:enc_end]
        past_tgt  = arr["tgt"][start:enc_end]
        past_time = arr["time"][start:enc_end]
        fut_cov   = arr["cov"][enc_end:dec_end]
        fut_vel   = arr["vel"][enc_end:dec_end]         # precomputed — no sklearn call
        fut_time  = arr["time"][enc_end:dec_end]
        tgt_norm  = arr["tgt"][enc_end:dec_end]

        if lookback < self.crop_hi:
            pad = self.crop_hi - lookback
            past_cov  = np.pad(past_cov,  [(pad, 0), (0, 0)])
            past_vel  = np.pad(past_vel,  [(pad, 0), (0, 0)])
            past_tgt  = np.pad(past_tgt,  [(pad, 0)])
            past_time = np.pad(past_time, [(pad, 0), (0, 0)])

        if self.use_aug and self.feat_drop_p > 0:
            mask = (np.random.rand(past_cov.shape[1]) > self.feat_drop_p).astype(np.float32)
            past_cov = past_cov * mask[None, :]

        return (
            torch.from_numpy(past_cov),
            torch.from_numpy(past_vel),
            torch.from_numpy(past_tgt),
            torch.from_numpy(past_time),
            torch.from_numpy(fut_cov.astype(np.float32)),
            torch.from_numpy(fut_vel),
            torch.from_numpy(fut_time.astype(np.float32)),
            torch.from_numpy(tgt_norm),
            torch.tensor(s, dtype=torch.long),
        )

print("Dataset class defined.")

Dataset class defined.


---
## §4. Model Architecture — LSTM Encoder-Decoder with Cross-Attention

Identical to 11a's `Seq2SeqGRU` except:
- `nn.GRU` → `nn.LSTM` in both encoder and decoder
- Decoder initialisation now requires both `h_0` and `c_0` (LSTM cell state)
- Cell state `c_0` is initialised from a second linear projection of the encoder's
  last-layer bidirectional cell state

In [6]:
class Seq2SeqLSTM(nn.Module):
    """Bi-LSTM encoder + Uni-LSTM decoder with cross-attention."""

    def __init__(self, n_domain, n_vel, n_time, n_series,
                 embed_dim, hidden_dim, n_enc_layers, n_dec_layers,
                 attn_heads, dropout):
        super().__init__()
        self.hidden_dim   = hidden_dim
        self.n_enc_layers = n_enc_layers
        self.series_emb   = nn.Embedding(n_series, embed_dim)

        enc_in_dim = n_domain + n_vel + 1 + n_time + embed_dim   # +1 for past_target
        dec_in_dim = n_domain + n_vel + n_time + embed_dim

        self.encoder = nn.LSTM(
            enc_in_dim, hidden_dim, n_enc_layers,
            batch_first=True, bidirectional=True,
            dropout=dropout if n_enc_layers > 1 else 0.0,
        )
        self.decoder = nn.LSTM(
            dec_in_dim, hidden_dim, n_dec_layers,
            batch_first=True, bidirectional=False,
            dropout=dropout if n_dec_layers > 1 else 0.0,
        )
        self.proj_enc_kv = nn.Linear(2 * hidden_dim, hidden_dim, bias=False)
        # Project concatenated fwd+bwd hidden/cell to decoder init
        self.proj_h = nn.Linear(2 * hidden_dim, hidden_dim)
        self.proj_c = nn.Linear(2 * hidden_dim, hidden_dim)
        self.cross_attn = nn.MultiheadAttention(
            hidden_dim, attn_heads, dropout=dropout, batch_first=True
        )
        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, past_cov, past_vel, past_tgt, past_time,
                fut_cov, fut_vel, fut_time, sid):
        B, L, _ = past_cov.shape
        H = fut_cov.shape[1]

        e     = self.series_emb(sid)
        e_enc = e.unsqueeze(1).expand(B, L, -1)
        e_dec = e.unsqueeze(1).expand(B, H, -1)

        enc_in = torch.cat(
            [past_cov, past_vel, past_tgt.unsqueeze(-1), past_time, e_enc], dim=-1
        )
        enc_out, (h_enc, c_enc) = self.encoder(enc_in)
        # h_enc, c_enc: (n_enc*2, B, hidden) — bidirectional

        # Decoder init from last encoder layer
        h_fwd = h_enc[-2];  h_bwd = h_enc[-1]
        c_fwd = c_enc[-2];  c_bwd = c_enc[-1]
        h_dec0 = self.proj_h(torch.cat([h_fwd, h_bwd], -1)).unsqueeze(0).contiguous()
        c_dec0 = self.proj_c(torch.cat([c_fwd, c_bwd], -1)).unsqueeze(0).contiguous()

        dec_in  = torch.cat([fut_cov, fut_vel, fut_time, e_dec], dim=-1)
        dec_out, _ = self.decoder(dec_in, (h_dec0, c_dec0))   # (B, H, hidden)

        enc_kv  = self.proj_enc_kv(enc_out)                    # (B, L, hidden)
        ctx, _  = self.cross_attn(dec_out, enc_kv, enc_kv)    # (B, H, hidden)

        out = dec_out + ctx
        return self.head(out).squeeze(-1)                       # (B, H)


def make_model():
    return Seq2SeqLSTM(
        n_domain=N_DOMAIN, n_vel=N_VELS, n_time=N_TIME, n_series=N_SERIES,
        embed_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM,
        n_enc_layers=N_ENC_LAYERS, n_dec_layers=N_DEC_LAYERS,
        attn_heads=ATTN_HEADS, dropout=DROPOUT,
    ).to(DEVICE)

tmp = make_model()
print(f"LSTM parameters: {sum(p.numel() for p in tmp.parameters() if p.requires_grad):,}")
del tmp

LSTM parameters: 819,073


---
## §5. Training & Evaluation Utilities (same as 11a)

In [7]:
import time

def train_one_epoch(model, loader, optimizer, amp_scaler=None):
    model.train()
    total, n = 0.0, 0
    for batch in loader:
        pc, pv, pt, ptm, fc, fv, ftm, tgt, sid = [x.to(DEVICE, non_blocking=True) for x in batch]
        optimizer.zero_grad(set_to_none=True)
        if amp_scaler is not None:
            with torch.cuda.amp.autocast():
                pred = model(pc, pv, pt, ptm, fc, fv, ftm, sid)
                loss = F.l1_loss(pred, tgt)
            amp_scaler.scale(loss).backward()
            amp_scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            amp_scaler.step(optimizer)
            amp_scaler.update()
        else:
            pred = model(pc, pv, pt, ptm, fc, fv, ftm, sid)
            loss = F.l1_loss(pred, tgt)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
        total += loss.item() * len(sid); n += len(sid)
    return total / n


@torch.no_grad()
def eval_fold(model, fold_k, feat_scaler, vel_scaler, mu, sd):
    model.eval()
    te_start, te_end = fold_test_slice(fold_k)
    lookback = min(ENCODER_LOOKBACK, te_start)

    all_preds, all_tgts, all_ts, all_sids = [], [], [], []
    for s in range(N_SERIES):
        arr    = SERIES_ARR[s]
        enc_s  = te_start - lookback
        cov_n  = np.nan_to_num(feat_scaler.transform(arr["cov"]), nan=0.0).astype(np.float32)
        vel_n  = np.nan_to_num(vel_scaler.transform(arr["vel"]), nan=0.0).astype(np.float32)
        tgt_n  = ((arr["tgt"] - mu[s]) / sd[s]).astype(np.float32)

        past_cov  = torch.from_numpy(cov_n[enc_s:te_start]).unsqueeze(0).to(DEVICE)
        past_vel  = torch.from_numpy(vel_n[enc_s:te_start]).unsqueeze(0).to(DEVICE)
        past_tgt  = torch.from_numpy(tgt_n[enc_s:te_start]).unsqueeze(0).to(DEVICE)
        past_time = torch.from_numpy(arr["time"][enc_s:te_start].astype(np.float32)).unsqueeze(0).to(DEVICE)

        fut_cov_n = np.nan_to_num(feat_scaler.transform(arr["cov"][te_start:te_end]), nan=0.0).astype(np.float32)
        fut_vel_n = np.nan_to_num(vel_scaler.transform(arr["vel"][te_start:te_end]), nan=0.0).astype(np.float32)

        fut_cov  = torch.from_numpy(fut_cov_n).unsqueeze(0).to(DEVICE)
        fut_vel  = torch.from_numpy(fut_vel_n).unsqueeze(0).to(DEVICE)
        fut_time = torch.from_numpy(arr["time"][te_start:te_end].astype(np.float32)).unsqueeze(0).to(DEVICE)
        sid_t    = torch.tensor([s], dtype=torch.long, device=DEVICE)

        pred_n = model(past_cov, past_vel, past_tgt, past_time, fut_cov, fut_vel, fut_time, sid_t)
        pred   = pred_n.cpu().numpy()[0] * sd[s] + mu[s]
        all_preds.append(pred)
        all_tgts.append(arr["tgt"][te_start:te_end])
        all_ts.append(arr["ts"][te_start:te_end])
        all_sids.append(s)

    pm = np.stack(all_preds); tm = np.stack(all_tgts); tsm = np.stack(all_ts)
    return wape(tm.flatten(), pm.flatten()), pm, tm, tsm, all_sids


def train_one_seed(fold_k, feat_scaler, vel_scaler, mu, sd, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    ds = Seq2SeqDataset(
        fold_k, feat_scaler, vel_scaler, mu, sd,
        use_aug=USE_AUG, feat_drop_p=FEAT_DROP_P if USE_AUG else 0.0,
        oversample_factor=HARD_SERIES_OVS if USE_AUG else 1,
    )
    loader = DataLoader(
        ds, BATCH_SIZE, shuffle=True,
        num_workers=N_WORKERS, pin_memory=(DEVICE.type == "cuda"),
        persistent_workers=(N_WORKERS > 0),
    )
    model = make_model()
    try:
        model = torch.compile(model)
    except Exception:
        pass
    opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sch   = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=N_EPOCHS, eta_min=LR_MIN)
    amp   = torch.cuda.amp.GradScaler() if DEVICE.type == "cuda" else None
    best_wape, no_imp, best_state = float("inf"), 0, None
    for ep in range(N_EPOCHS):
        train_one_epoch(model, loader, opt, amp)
        sch.step()
        w, _, _, _, _ = eval_fold(model, fold_k, feat_scaler, vel_scaler, mu, sd)
        if w < best_wape:
            best_wape  = w
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_imp = 0
        else:
            no_imp += 1
        if no_imp >= PATIENCE:
            break
    model.load_state_dict(best_state)
    return eval_fold(model, fold_k, feat_scaler, vel_scaler, mu, sd)

print("Utilities defined.")

Utilities defined.


---
## §6. Full LSTM CV — 3 folds × 5 seeds

In [8]:
oof_rows = []
per_fold_wape = []
per_fold_spread = []

print(f"Full CV: {N_FOLDS} folds × {N_SEEDS} seeds (LSTM)")
print(f"Config: use_aug={USE_AUG}, use_velocity={USE_VELOCITY}")

for k in range(N_FOLDS):
    feat_s, vel_s, mu_k, sd_k = fit_fold_normalizers(k)
    te_start, te_end = fold_test_slice(k)

    seed_preds = []
    for seed in SEQ_SEEDS:
        t0 = time.time()
        w, preds, tgts, ts_mat, sids = train_one_seed(k, feat_s, vel_s, mu_k, sd_k, seed)
        seed_preds.append(preds)
        print(f"  fold={k}, seed={seed}: WAPE={w:.3f} ({time.time()-t0:.0f}s)")

    seed_arr = np.stack(seed_preds)
    avg_pred = seed_arr.mean(axis=0)
    seed_wapes = [wape(tgts.flatten(), seed_arr[i].flatten()) for i in range(N_SEEDS)]
    spread = max(seed_wapes) - min(seed_wapes)
    fold_w = wape(tgts.flatten(), avg_pred.flatten())
    per_fold_wape.append(fold_w)
    per_fold_spread.append(spread)

    for s_idx, s_code in enumerate(sids):
        sid_str = code_to_sid[s_code]
        for h in range(EVAL_HORIZON):
            oof_rows.append({
                "series_id":  sid_str,
                "timestamp":  ts_mat[s_idx, h],
                "target":     float(tgts[s_idx, h]),
                "pred_lstm":  float(avg_pred[s_idx, h]),
                "lead_h":     h + 1,
                "fold":       k,
            })
    print(f"  ── fold {k}: avg WAPE={fold_w:.3f}, seed spread={spread:.3f}")

oof_df = pd.DataFrame(oof_rows)
lstm_arr = np.array(per_fold_wape)
LSTM_WAPE = lstm_arr.mean()
LSTM_STD  = lstm_arr.std()

print()
print(f"{'fold':>6}  {'WAPE':>8}  {'spread':>8}")
for k, (w, sp) in enumerate(zip(lstm_arr, per_fold_spread)):
    print(f"  {k:4d}    {w:>7.3f}    {sp:>7.3f}")
print(f"  ────")
print(f"  mean    {LSTM_WAPE:>7.3f}")
print(f"  std     {LSTM_STD:>7.3f}")

oof_df.to_csv(OOF_OUT, index=False)
print(f"\nOOF saved to {OOF_OUT}")

Full CV: 3 folds × 5 seeds (LSTM)
Config: use_aug=True, use_velocity=True
  fold=0, seed=42: WAPE=14.096 (117s)
  fold=0, seed=7: WAPE=14.233 (125s)
  fold=0, seed=13: WAPE=14.186 (92s)
  fold=0, seed=99: WAPE=14.065 (125s)
  fold=0, seed=1234: WAPE=14.080 (99s)
  ── fold 0: avg WAPE=13.817, seed spread=0.168
  fold=1, seed=42: WAPE=15.667 (131s)
  fold=1, seed=7: WAPE=15.838 (117s)
  fold=1, seed=13: WAPE=15.476 (139s)
  fold=1, seed=99: WAPE=15.613 (133s)
  fold=1, seed=1234: WAPE=15.427 (125s)
  ── fold 1: avg WAPE=15.248, seed spread=0.411
  fold=2, seed=42: WAPE=14.510 (130s)
  fold=2, seed=7: WAPE=14.479 (129s)
  fold=2, seed=13: WAPE=14.433 (129s)
  fold=2, seed=99: WAPE=14.518 (128s)
  fold=2, seed=1234: WAPE=14.388 (136s)
  ── fold 2: avg WAPE=14.169, seed spread=0.130

  fold      WAPE    spread
     0     13.817      0.168
     1     15.248      0.411
     2     14.169      0.130
  ────
  mean     14.411
  std       0.609

OOF saved to /kaggle/working/oof_predictions_11b_lst

---
## §7. Residual Correlation vs exp-7c Blend

In [9]:
oof_7c_path = Path("/kaggle/input/datasets/xitij27/tempdata/oof_predictions_7b_blend.csv")
if not oof_7c_path.exists():
    oof_7c_path = Path("/kaggle/input/datasets/xitij27/baseline-code/experiment-7/oof_predictions_7b_blend.csv")

if oof_7c_path.exists():
    oof_7c = pd.read_csv(oof_7c_path, parse_dates=["timestamp"])
    pred_col_7c = "pred_blend" if "pred_blend" in oof_7c.columns else oof_7c.columns[-1]
    joined = oof_df.merge(
        oof_7c[["series_id", "timestamp", pred_col_7c, "fold"]],
        on=["series_id", "timestamp", "fold"], how="inner"
    )
    joined["resid_lstm"] = joined["target"] - joined["pred_lstm"]
    joined["resid_7c"]   = joined["target"] - joined[pred_col_7c]
    r_overall = np.corrcoef(joined["resid_lstm"], joined["resid_7c"])[0, 1]
    r_per_fold = [
        np.corrcoef(joined.loc[joined["fold"]==k, "resid_lstm"],
                    joined.loc[joined["fold"]==k, "resid_7c"])[0, 1]
        for k in range(N_FOLDS)
    ]
    print("=== Pearson r(residual_LSTM, residual_7c-blend) ===")
    for k, r in enumerate(r_per_fold):
        print(f"  fold {k}: r = {r:.4f}")
    print(f"  pooled:  r = {r_overall:.4f}")
    if r_overall < 0.85:
        print("→ Strong diversity. LSTM is a valuable blend arm.")
    elif r_overall < 0.90:
        print("→ Moderate diversity. Worth blending at OOF-optimised weight.")
    else:
        print("→ Limited diversity (r ≥ 0.90).")
else:
    print(f"exp-7c OOF not found. Skipping correlation analysis.")

=== Pearson r(residual_LSTM, residual_7c-blend) ===
  fold 0: r = 0.9764
  fold 1: r = 0.9645
  fold 2: r = 0.9749
  pooled:  r = 0.9720
→ Limited diversity (r ≥ 0.90).


---
## §8. Per-Horizon WAPE & Per-Series WAPE

In [10]:
BINS = [(1, 1), (2, 6), (7, 24), (25, 72), (73, 168), (169, 336)]
print(f"{'lead-h range':>14}  {'count':>8}  {'WAPE LSTM':>10}")
for lo, hi in BINS:
    mask = (oof_df["lead_h"] >= lo) & (oof_df["lead_h"] <= hi)
    sub  = oof_df[mask]
    w    = wape(sub["target"], sub["pred_lstm"])
    print(f"  {lo:>3}..{hi:<3}h  {len(sub):>8d}    {w:>7.3f}")

per_series = oof_df.groupby("series_id").apply(
    lambda g: pd.Series({"wape_lstm": wape(g["target"], g["pred_lstm"])})
).reset_index().sort_values("wape_lstm", ascending=False)
print(f"\nPer-series: min={per_series['wape_lstm'].min():.2f}, max={per_series['wape_lstm'].max():.2f}")

  lead-h range     count   WAPE LSTM
    1..1  h       288     19.340
    2..6  h      1440     20.090
    7..24 h      5184     12.538
   25..72 h     13824     16.521
   73..168h     27648     13.593
  169..336h     48384     14.403

Per-series: min=7.33, max=25.39


---
## §9. Validation Predictions

In [11]:
@torch.no_grad()
def predict_validation(model, feat_scaler, vel_scaler, mu, sd):
    model.eval()
    all_preds = []
    lookback  = ENCODER_LOOKBACK
    for s in range(N_SERIES):
        arr   = SERIES_ARR[s]
        cov_n = np.nan_to_num(feat_scaler.transform(arr["cov"]), nan=0.0).astype(np.float32)
        vel_n = np.nan_to_num(vel_scaler.transform(arr["vel"]), nan=0.0).astype(np.float32)
        tgt_n = ((arr["tgt"] - mu[s]) / sd[s]).astype(np.float32)
        enc_s = T_PER_SERIES - lookback
        past_cov  = torch.from_numpy(cov_n[enc_s:]).unsqueeze(0).to(DEVICE)
        past_vel  = torch.from_numpy(vel_n[enc_s:]).unsqueeze(0).to(DEVICE)
        past_tgt  = torch.from_numpy(tgt_n[enc_s:]).unsqueeze(0).to(DEVICE)
        past_time = torch.from_numpy(arr["time"][enc_s:].astype(np.float32)).unsqueeze(0).to(DEVICE)
        val   = VAL_ARR[s]
        fc_n  = np.nan_to_num(feat_scaler.transform(val["cov"]), nan=0.0).astype(np.float32)
        fv_n  = np.nan_to_num(vel_scaler.transform(val["vel"]), nan=0.0).astype(np.float32)
        fut_cov  = torch.from_numpy(fc_n).unsqueeze(0).to(DEVICE)
        fut_vel  = torch.from_numpy(fv_n).unsqueeze(0).to(DEVICE)
        fut_time = torch.from_numpy(val["time"].astype(np.float32)).unsqueeze(0).to(DEVICE)
        sid_t    = torch.tensor([s], dtype=torch.long, device=DEVICE)
        pred = model(past_cov, past_vel, past_tgt, past_time, fut_cov, fut_vel, fut_time, sid_t)
        all_preds.append((pred.cpu().numpy()[0] * sd[s] + mu[s]))
    return np.stack(all_preds)


all_cov_full = np.concatenate([SERIES_ARR[s]["cov"] for s in range(N_SERIES)])
all_vel_full = np.concatenate([SERIES_ARR[s]["vel"] for s in range(N_SERIES)])
feat_s_full  = RobustScaler().fit(all_cov_full)
vel_s_full   = RobustScaler().fit(all_vel_full)
mu_full = np.array([SERIES_ARR[s]["tgt"].mean() for s in range(N_SERIES)], dtype=np.float32)
sd_full = np.array([max(float(SERIES_ARR[s]["tgt"].std()), 1e-3) for s in range(N_SERIES)], dtype=np.float32)

# end_idx_override=T_PER_SERIES: train on ALL data (windows up to start=3648, decoder target up to 4320)
# This is the critical fix — fold_k=0 alone only exposes hours 0..3312 to the model.
FINAL_END_IDX = T_PER_SERIES

val_seed_preds = []
for seed in SEQ_SEEDS:
    torch.manual_seed(seed); np.random.seed(seed)
    ds_full  = Seq2SeqDataset(
        fold_k=0,
        feat_scaler=feat_s_full, vel_scaler=vel_s_full,
        mu=mu_full, sd=sd_full,
        use_aug=USE_AUG, feat_drop_p=FEAT_DROP_P if USE_AUG else 0.0,
        oversample_factor=HARD_SERIES_OVS if USE_AUG else 1,
        end_idx_override=FINAL_END_IDX,
    )
    loader_f = DataLoader(
        ds_full, BATCH_SIZE, shuffle=True,
        num_workers=N_WORKERS, pin_memory=(DEVICE.type == "cuda"),
        persistent_workers=(N_WORKERS > 0),
    )
    model_f  = make_model()
    try:
        model_f = torch.compile(model_f)
    except Exception:
        pass
    opt_f    = torch.optim.AdamW(model_f.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sch_f    = torch.optim.lr_scheduler.CosineAnnealingLR(opt_f, N_EPOCHS, eta_min=LR_MIN)
    amp_f    = torch.cuda.amp.GradScaler() if DEVICE.type == "cuda" else None
    for _ in range(N_EPOCHS):
        train_one_epoch(model_f, loader_f, opt_f, amp_f); sch_f.step()
    vp = predict_validation(model_f, feat_s_full, vel_s_full, mu_full, sd_full)
    val_seed_preds.append(vp)
    print(f"  seed {seed} done")

val_avg = np.stack(val_seed_preds).mean(axis=0)
val_rows = [
    {"series_id": code_to_sid[s], "timestamp": VAL_ARR[s]["ts"][h], "prediction": float(val_avg[s, h])}
    for s in range(N_SERIES) for h in range(EVAL_HORIZON)
]
pd.DataFrame(val_rows).to_csv(VAL_OUT, index=False)
print(f"Saved to {VAL_OUT}")

  seed 42 done
  seed 7 done
  seed 13 done
  seed 99 done
  seed 1234 done
Saved to /kaggle/working/validation_predictions_11b_lstm.csv


---
## §10. GRU vs LSTM Comparison

In [12]:
print("=" * 55)
print("Experiment 11b — LSTM Encoder-Decoder Summary")
print("=" * 55)
print(f"CV WAPE per fold  : {[f'{w:.3f}' for w in lstm_arr]}")
print(f"CV WAPE mean ± std: {LSTM_WAPE:.3f} ± {LSTM_STD:.3f}")
if 'r_overall' in dir():
    print(f"Residual r vs 7c  : {r_overall:.4f}")
print()
print("Next step: open 11c-blend.ipynb with both OOF CSVs to compute ensemble WAPE.")

Experiment 11b — LSTM Encoder-Decoder Summary
CV WAPE per fold  : ['13.817', '15.248', '14.169']
CV WAPE mean ± std: 14.411 ± 0.609
Residual r vs 7c  : 0.9720

Next step: open 11c-blend.ipynb with both OOF CSVs to compute ensemble WAPE.
